
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>


# Securing Data in Unity Catalog

In this lab, you will create two examples of fine grain access control:

1. Dynamic Views using Role-Based Access Control (RBAC)

2. Row Filtering and Column Masking Tables using Role-Based Access Control (RBAC)

3. Dynamic Views Using Attribute-Based Access Control (ABAC)

4. Row Filtering and Column Masking Tables using Role-Based Access Control (RBAC) 

### Learning Objectives
By the end of this demo, you will be able to:
1. Assign access permissions to the newly created view for account users and execute queries to validate the view's functionality and data integrity.
1. Develop a dynamic view of the cloned table, applying data redaction and access restrictions to enhance data security.

## REQUIRED - SELECT SERVERLESS

Before executing cells in this notebook, please select Serverless cluster in the lab. Be aware that **Serverless** is enabled by default.

Follow these steps to select the Serverless cluster:

- Navigate to the top-right of this notebook and click the drop-down menu to select your cluster. By default, the notebook will use **Serverless**.

## A. Classroom Setup

Run the following cell to configure your working environment for this course. It will also set your default catalog to your specific catalog and the schema to the schema name shown below using the `USE` statements.
<br></br>


```
USE CATALOG <your catalog>;
USE SCHEMA <your catalog>.<schema>;
```

**NOTE:** The `DA` object is only used in Databricks Academy courses and is not available outside of these courses. It will dynamically reference the information needed to run the course.

Run the code below and confirm that your current catalog is set to your unique catalog name and that the current schema is **default**.


In [0]:
Use office.hr;

In [0]:
SELECT current_catalog(), current_schema()

## B. Protect Columns and Rows with Column Masking and Row Filtering

### REQUIRED: Create the Table users
1. Run the code below to create the **users** table in your **default** schema.

In [0]:
CREATE OR REPLACE TABLE users AS
SELECT *
FROM samples.wanderbricks.users;

2. Run a query to view *10* rows from the **users** table in your **default** schema. Notice that the table contains information such as **name**, **email**, and **country**.

In [0]:
--<FILL_IN>
SELECT
  name,
  email,
  country
FROM
  users
LIMIT 10;


### B1. Create a Function to Perform Column Masking
View the [Filter sensitive table data using row filters and column masks](https://docs.databricks.com/en/tables/row-and-column-filters.html) documentation for additional help.
1. Create a function named **email_mask** that redacts the **email** column in the **users** table if the user is not a member of the ('admin') group using the `is_account_group_member` function. The **email_mask** function should return the string *REDACTED EMAIL* if the user is not a member.

In [0]:
--<FILL_IN>
CREATE OR REPLACE FUNCTION email_mask (email STRING)
RETURNS STRING
RETURN 
  CASE 
    WHEN 
      is_account_group_member('admin') 
    THEN email 
    ELSE 'REDACTED EMAIL' 
  END

2. Apply the column masking function **email_mask** to the **users** table using the `ALTER TABLE` statement.

In [0]:
--<FILL_IN>
ALTER TABLE users 
ALTER COLUMN email 
SET MASK email_mask;

3. Run the cell below to confirm that you have correctly masked the **email** column in the **users** table. If an error is returned make sure the function is created and applied correctly.

In [0]:
%python
sdf = spark.sql('SELECT email FROM users GROUP BY email')
value = sdf.collect()[0][0]

assert value == 'REDACTED EMAIL', 'The email column should only contain the value "REDACTED EMAIL". Please create the correct column mask function and apply it to the users table.'
print('You correctly applied the column mask to the users table.')

4. Run the query below to view the **users** table with the column mask applied. Confirm that the **email** column displays the value *REDACTED EMAIL*.



In [0]:
SELECT *
FROM users
LIMIT 10;

### B2. Create a Function to Perform Row Filtering
View the [Filter sensitive table data using row filters and column masks](https://docs.databricks.com/en/tables/row-and-column-filters.html) documentation for additional help.

1. Run the cell below to count the total number of rows in the **users** table. Confirm that the table contains 124,509 rows of data.


In [0]:
SELECT count(*) AS TotalRows
FROM users;

2. Create a function named **country_filter** that filters on **country** in the **users** table if the user is not a member of the ('admin') group using the `is_account_group_member` function. The function should only return rows where **country** equals *India*.

    View the [if function](https://docs.databricks.com/en/sql/language-manual/functions/if.html) documentation for additional help.

In [0]:
--India 23029
--<FILL_IN>
CREATE OR REPLACE FUNCTION country_filter (country STRING)
RETURNS BOOLEAN
RETURN if(
  is_account_group_member('admin') , 
  TRUE, 
  country = 'India');

3. Apply the function row filtering function `country_filter` to the **users** table using the `ALTER TABLE` statement.

In [0]:
--<FILL_IN>
ALTER TABLE users 
SET ROW FILTER country_filter on (country)

4. Run the cell below to confirm that you added row filtering to the **users** table column **country** correctly.


In [0]:
%python
sdf = spark.sql('SELECT country FROM users GROUP BY country')
value = sdf.collect()[0][0]

assert value == 'India', 'The country column should only contain the value India. Please create the correct row filter function and apply it to the users table.'
print('You correctly applied the row filter.')

5. Run the query below to count the number of rows in the **users** table for you since you've filtered out rows for users not admins. Confirm you can only view *23,029* rows (*where country = 'India'*).

In [0]:
SELECT count(*) AS TotalRows
FROM users;

6. Run the query below to view the **users** table. 

    Confirm the final table:
    - redactes the **email** column and
    - filters rows based on the **country** column for users who are not *admins*.

In [0]:
SELECT *
FROM users;

## C. Protecting Columns and Rows with Dynamic Views


### REQUIRED: Create the Table users_new
1. Run the code below to create the **users_new** table in your **default** schema.

In [0]:
CREATE OR REPLACE TABLE users_new AS
SELECT *
FROM samples.wanderbricks.users;

2. Run a query to view *10* rows from the **users_new** table in your **default** schema. Notice that the table contains information such as **name**, **email**, and **country**.

In [0]:
SELECT *
FROM users_new
LIMIT 10;

### C1. Create the Dynamic View
Let's create a view named **vw_users** that presents a processed view of the **users_new** table data with the following transformations:
- Selects all columns from the **users_new** table.

- Redact all values in the **email** column to *REDACTED EMAIL* unless you are in the `is_account_group_member('admins')`
    - HINT: Use a `CASE WHEN` statement in the `SELECT` clause.

- Restrict the rows where **country** is equal to *India* unless you are in the `is_account_group_member('admins')`.
    - HINT: Use a `CASE WHEN` statement in the `WHERE` clause.

In [0]:
--<FILL_IN>
CREATE OR REPLACE VIEW vw_users AS
SELECT
  CASE 
    WHEN 
      is_account_group_member('admin') 
    THEN email 
    ELSE 'REDACTED EMAIL' 
  END as email_clean,
  * EXCEPT (email)
FROM users_new
WHERE
  if(
  is_account_group_member('admin') , 
  TRUE, 
  country = 'India');

3. Run the cell below to confirm that you have correctly masked the **email** column in the **vw_users** view. If an error is returned make sure the function is created and applied correctly.

In [0]:
%python
sdf = spark.sql('SELECT email_clean FROM vw_users GROUP BY email_clean')
value = sdf.collect()[0][0]

assert value == 'REDACTED EMAIL', 'The email column should only contain the value "REDACTED EMAIL". Please use the following expression in the SELECT clause: CASE WHEN is_account_group_member("admins") THEN email ELSE "REDACTED EMAIL" END as email.'
print('You correctly applied the column mask to the vw_users view.')

4. Run the cell below to confirm that you added row filtering to the **users** table column **country** correctly.

In [0]:
%python
sdf = spark.sql('SELECT country FROM vw_users GROUP BY country')
value = sdf.collect()[0][0]

assert value == 'India', 'The country column should only contain the value India. Please create the correct row filter in the view using the following expression in the WHERE clause: CASE WHEN is_account_group_member("admins") THEN TRUE ELSE country = India END.'
print('You correctly applied the row filter to the view.')

5. Display the data in the **vw_users** view. Confirm the **email** column is redacted.

In [0]:
SELECT * 
FROM vw_users;

6. Count the number of rows in the **vw_users** view. Confirm the view contains *23,029* rows.

In [0]:
SELECT count(*)
FROM vw_users;

### C2. Issue Grant Access to View
1. Let us issue a grant for "account users" to view the **vw_users** view.

**NOTE:** You will also need to provide users access to the catalog and schema. In this shared training environment, you are unable to grant access to your catalog to other users.


In [0]:
--<FILL_IN>
GRANT SELECT ON CATALOG office to `account users`;
GRANT SELECT ON SCHEMA hr to `account users`;
GRANT SELECT ON VIEW vw_users to `account users`;

2. Use the `SHOW` statement to displays all privileges (inherited, denied, and granted) that affect the **vw_users** view. Confirm that the **Principal** column contains *account users*.

    View the [SHOW GRANTS](https://docs.databricks.com/en/sql/language-manual/security-show-grant.html) documentation for help.

In [0]:
--<FILL_IN>
SHOW GRANTS ON vw_users;

## D. Using Governance Tag with Attribute-Based Access Control

### REQUIRED: Create the Table users_abac
1. Run the code below to create the **users_abac** table in your **default** schema.

In [0]:
--<FILL_IN>
CREATE OR REPLACE TABLE users_abac AS
SELECT *
FROM samples.wanderbricks.users;

###D1. Create the Tags on PII Columns

1. Create a `pii` tag with the value `email` on the **email** column in the **users_abac** table 

2. Create a `class:location` tag with no value on the **country** column in the **users_abac** table 

These tags exist by default in most configurations of Unity Catalog. For more information on how to customize tags.

View the [Create and manage governed tags](https://docs.databricks.com/aws/en/admin/governed-tags/manage-governed-tags) documentation for additional help.

In [0]:
--<FILL_IN>
-- Add governed tag to email column
ALTER TABLE users_abac
ALTER COLUMN email
SET TAGS ('pii' = 'email');


In [0]:
--<FILL_IN>
-- Add governed tag to country column
ALTER TABLE users_abac
ALTER COLUMN country
SET TAGS ('class.location');

###D2. Create the Policies on The Columns with Governed Tags
View the [Create and manage attribute-based access control (ABAC) policies](https://docs.databricks.com/aws/en/data-governance/unity-catalog/abac/policies?language=SQL) documentation for additional help.
1. Create a policy named **abac_column_mask** that redacts any columns with the `email` value on a `pii` tag using the **email_mask** function.
2. Create a policy named **abac_row_filter** that filters any value with a `class.location` tag  using the **country_filter** function.

In [0]:
--<FILL_IN>
-- Add row column mask tag to table
CREATE OR REPLACE POLICY abac_column_mask
ON TABLE users_abac
COMMENT 'Simple column mask on table via policy'
COLUMN MASK email_mask
TO `account users`
FOR TABLES
MATCH COLUMNS
  hasTagValue('classification','email') AS name_col
ON COLUMN name_col

In [0]:
--<FILL_IN>
-- Add row filter policy tag to table
CREATE OR REPLACE POLICY abac_row_filter
ON TABLE users_abac
COMMENT 'Simple row filter on table via policy'
ROW FILTER country_filter
TO `account users`
FOR TABLES
MATCH COLUMNS
  hasTag('class.location') AS country_col
USING COLUMNS (country_col)

###D3. Validate the Policies

1. View the policies via SQL
1. Verify the policies work

In [0]:
%python
sdf = spark.sql('SELECT country FROM users_abac GROUP BY country')
value = sdf.collect()[0][0]

assert value == 'India', 'The country column should only contain the value India. Please create the correct policy using the previous row filter'
print('You correctly applied the row filter to the table using a policy.')

In [0]:
%python
sdf = spark.sql('SELECT email FROM users_abac GROUP BY email')
value = sdf.collect()[0][0]

assert value == 'REDACTED EMAIL', 'The email column should only contain the value "REDACTED EMAIL". Please create the correct policy using the previous column mask function.'
print('You correctly applied the column mask to the table using a policy.')

In [0]:
--<FILL_IN>
select * from users_abac

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>